# Indonesia Provincial GVA — Visualisations
**Input:** `03_01_provincial_gva.csv` — output of `03_01_provincial_gva_calculations.ipynb`

All growth charts use **ADHK** (constant 2010 prices).  
`plot_gva_pct_gdp` uses **ADHB** (current prices) for structural composition.


## 1. Packages

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from pathlib import Path
from ipywidgets import interact



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Load data

In [2]:

INTER = Path(r'C:\Users\Admin\OneDrive\Desktop\Personal Projects\Indonesia GVA\02_intermediate_data')

# Annual: both ADHK and ADHB — used for structural charts and annual YoY
gva_ann = pd.read_csv(INTER / '03_01_provincial_gva_annual.csv')

# Quarterly: ADHK only — used for same-quarter YoY and contribution charts
gva_qtr = pd.read_csv(INTER / '03_01_provincial_gva_quarterly.csv')

print('Annual rows  :', len(gva_ann), '| bases:', gva_ann['price_basis'].value_counts().to_dict())
print('Quarterly rows:', len(gva_qtr))
print('Quarterly periods:', sorted(gva_qtr['period'].unique()))

Annual rows  : 9044 | bases: {'ADHB': 4522, 'ADHK': 4522}
Quarterly rows: 16150
Quarterly periods: ['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']


## 3. Rebuild working tables

Reconstruct the same DataFrames the calculation notebook produced,
directly from the exported CSV.


In [ ]:
# Annual Tables
ann_adhk = gva_ann[gva_ann['price_basis'] == 'ADHK'].copy() # Get annual data for CONSTANT (real regional GDP) prices
ann_adhb = gva_ann[gva_ann['price_basis'] == 'ADHB'].copy() # Get annual data for CURRENT (nominal regional GDP) prices, used for contribution charts and deflator calculations

# Create a provincial constant price table
prov_adhk_ann = (
    ann_adhk # original ADHK df 
    .dropna(subset=['gva_sector']) # drop rows where GVA is NA
    .groupby(['provinsi', 'year']) # group by province and year
    .agg(
        gva_total          = ('gva_sector', 'sum') # aggregates sectoral GVA to get total GVA per province-year
    )
    .reset_index()  # drops province-year labels back into its values
    .sort_values(['provinsi', 'year']) # sort rows by province alphabetically, then by year ascending (earliest year first)
)

prov_adhk_ann.head(10)

,provinsi,year,gva_total
0,Aceh,2020,"131,277.4155"
1,Aceh,2021,"134,935.7657"
2,Aceh,2022,"140,622.1671"
3,Aceh,2023,"146,589.7572"
4,Aceh,2024,"153,386.1561"
5,Aceh,2025,"157,948.8233"
6,Bali,2020,"146,637.5452"
7,Bali,2021,"143,057.7963"
8,Bali,2022,"149,932.9162"
9,Bali,2023,"158,476.3270"


In [51]:
prov_adhk_ann['period']                   = (prov_adhk_ann['year']
                                             .astype(str)) # convert year integer into string so it can be used as a label downstream

prov_adhk_ann['gva_total_yoy']            = (prov_adhk_ann
                                             .groupby('provinsi')['gva_total'] # group by province and total gva
                                             .pct_change(fill_method = None) * 100 # get % yoy change, without filling missing values, since we already sorted the dfs this should be correct
                                            )

ann_adhk['period']      = ann_adhk['year'].astype(str) 

ann_adhb_sorted         = ann_adhb.copy()

ann_adhk[ann_adhk['year'] != 2020].drop_duplicates(subset = ['provinsi', 'year']).head(10)

,provinsi,year,sector_code,sector_name,sector_short,gva_sector,gva_yoy,price_basis,gva_pct_gdp,period
4539,Aceh,2021,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"38,268.0639",-0.3468,ADHK,NaN,2021
4556,Aceh,2022,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"39,521.4501",3.2753,ADHK,NaN,2022
4573,Aceh,2023,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"42,177.3914",6.7203,ADHK,NaN,2023
4590,Aceh,2024,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"42,679.0053",1.1893,ADHK,NaN,2024
4607,Aceh,2025,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"44,215.2877",3.5996,ADHK,NaN,2025
4624,Aceh,2026,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,NaN,NaN,ADHK,NaN,2026
4658,Bali,2021,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"21,620.9398",0.3256,ADHK,NaN,2021
4675,Bali,2022,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"21,437.3815",-0.8490,ADHK,NaN,2022
4692,Bali,2023,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"21,337.4863",-0.4660,ADHK,NaN,2023
4709,Bali,2024,A,"Pertanian, Kehutanan dan Perikanan",Pertanian,"21,872.3243",2.5066,ADHK,NaN,2024


In [ ]:

# ── Quarterly tables ──────────────────────────────────────────────────────────
qtr_adhk = gva_qtr.copy() # get quarterly real regional GDP data

prov_adhk_qtr = (
    qtr_adhk # original df
    .dropna(subset=['gva_sector']) # drow rows where sectoral GVA is NA
    .groupby(['provinsi', 'year', 'quarter', 'period']) # group by province, year, quarter, and period (e.g. 2021Q1)
    .agg(
        gva_total          = ('gva_sector', 'sum'), # get sum of sectoral GVA to get total GVA per province-year-quarter
    )
    .reset_index() # drop province-year-quarter labels back into its values
    .sort_values(['provinsi', 'year', 'quarter']) # sort rows by province alphabetically, then by year ascending (earliest year first), then by quarter ascending (Q1, Q2, Q3, Q4)
)

# add additional columns for period labels and YoY growth calculations
prov_adhk_qtr['gva_total_yoy'] = ( 
    prov_adhk_qtr # original df
    .groupby(['provinsi', 'quarter'])['gva_total']  # group by province and quarter, so that we compare the same quarter of the previous year
    .pct_change(fill_method=None) * 100 # get pct_change, since we sorted the df by province, year, and quarter in ascending order, this should give us the correct YoY change
)

# ── Period lists ──────────────────────────────────────────────────────────────
province_list   = sorted(qtr_adhk['provinsi'].dropna().unique())
all_periods_ann = sorted(ann_adhk['period'].dropna().unique())      # ['2021', '2022', ...]
all_periods_qtr = sorted(qtr_adhk['period'].dropna().unique())      # ['2021Q1', '2021Q2', ...]


print(f'Provinces       : {len(province_list)}')
print(f'Annual periods  : {all_periods_ann}')
print(f'Quarterly periods: {all_periods_qtr}')
print(prov_adhk_qtr[prov_adhk_qtr['year'] != 2020].head(10))

# Now we have three datasets

# ann_adhk is annual real GVA data for each province-sector-year combination, used for:

# 1. Structural composition chart - what share of the provincial real GDP each sector represents
# 2. Sectoral GVA YOY vs. GDRP YOY - to see which sectors are driving growth or contraction in each province

# ann_adhb_sorted is annual nominal GVA data for each province-sector-year combination, used for:

# 1. Contribution to growth charts - to see which sectors are contributing the most to growth or contraction in each province

# prov_adhk_qtr is quarterly real GVA data for each province-year-quarter combination, with YoY growth rates calculated

# This is used for:
# 1. Quarterly provincial GVA growth (YoY) vs. regional real GDP growth

Provinces       : 38
Annual periods  : ['2020', '2021', '2022', '2023', '2024', '2025', '2026']
Quarterly periods: ['2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']
   provinsi  year quarter  period   gva_total  gva_total_yoy
4      Aceh  2021      Q1  2021Q1 32,014.8592        -1.8940
5      Aceh  2021      Q2  2021Q2 33,027.5336         2.5595
6      Aceh  2021      Q3  2021Q3 34,320.8967         3.0127
7      Aceh  2021      Q4  2021Q4 35,572.4762         7.3918
8      Aceh  2022      Q1  2022Q1 33,380.1449         4.2645
9      Aceh  2022      Q2  2022Q2 34,495.3973         4.4444
10     Aceh  2022      Q3  2022Q3 35,175.0361         2.4887
11     Aceh  2022      Q4  2022Q4 37,571.5888         5.6198
12     Aceh  2023      Q1  2023Q1 34,932.2917         4.6499
13     Aceh  2023      Q

## 4. Palettes & constants

In [ ]:
# Shorten sector names for better readability in charts

SECTOR_SHORT = {
    'Pertanian, Kehutanan dan Perikanan'                                    : 'Pertanian',
    'Pertambangan dan Penggalian'                                           : 'Pertambangan',
    'Industri Pengolahan'                                                   : 'Industri Pengolahan',
    'Pengadaan Listrik dan Gas'                                             : 'Listrik & Gas',
    'Pengadaan Air, Pengelolaan Sampah, Limbah dan Daur Ulang'             : 'Air & Sampah',
    'Konstruksi'                                                            : 'Konstruksi',
    'Perdagangan Besar dan Eceran, Reparasi Mobil dan Sepeda Motor'        : 'Perdagangan',
    'Transportasi dan Pergudangan'                                          : 'Transportasi',
    'Penyediaan Akomodasi dan Makan Minum'                                 : 'Akomodasi',
    'Informasi dan Komunikasi'                                              : 'Infokom',
    'Jasa Keuangan dan Asuransi'                                           : 'Keuangan',
    'Real Estate'                                                           : 'Real Estate',
    'Jasa Perusahaan'                                                       : 'Jasa Perusahaan',
    'Administrasi Pemerintahan, Pertahanan dan Jaminan Sosial Wajib'       : 'Adm. Pemerintahan',
    'Jasa Pendidikan'                                                       : 'Pendidikan',
    'Jasa Kesehatan dan Kegiatan Sosial'                                   : 'Kesehatan',
    'Jasa Lainnya'                                                          : 'Jasa Lainnya',
    'Produk Domestik Regional Bruto'                                        : 'PDRB Total',
}

# Assign colours for sectors 

SECTOR_COLOURS = {
    'Pertanian'         : '#2D6A4F',
    'Pertambangan'      : '#B5838D',
    'Industri Pengolahan': '#457B9D',
    'Listrik & Gas'     : '#E9C46A',
    'Air & Sampah'      : '#84A98C',
    'Konstruksi'        : '#E76F51',
    'Perdagangan'       : '#264653',
    'Transportasi'      : '#6D6875',
    'Akomodasi'         : '#F4A261',
    'Infokom'           : '#2A9D8F',
    'Keuangan'          : '#E63946',
    'Real Estate'       : '#A8DADC',
    'Jasa Perusahaan'   : '#9B2226',
    'Adm. Pemerintahan' : '#AE2012',
    'Pendidikan'        : '#0A9396',
    'Kesehatan'         : '#94D2BD',
    'Jasa Lainnya'      : '#CA6702',
}

# Assign colours for years (for quarterly charts, to maintain consistency across provinces)
YEAR_COLOURS = {
    2021: '#185FA5',
    2022: '#378ADD',
    2023: '#85B7EB',
    2024: '#B5D4F4',
    2025: '#D6EAF8',
}

# Apply sector_short if not already in the table
if 'sector_short' not in qtr_adhk.columns:
    qtr_adhk['sector_short']     = qtr_adhk['sector_name'].map(SECTOR_SHORT)
if 'sector_short' not in ann_adhb_sorted.columns:
    ann_adhb_sorted['sector_short'] = ann_adhb_sorted['sector_name'].map(SECTOR_SHORT)

## 5. PDRB benchmarks

In [47]:
pdrb_long = pd.read_csv(INTER / '02_01_pdrb_sectoral_long.csv') # long format province-sector real and nominal GDP data

df_adhk_pdrb = pdrb_long[pdrb_long['price_basis'] == 'ADHK'].copy() # long-format province-sector real GDP data

In [ ]:

pdrb_prov_yoy = (
    df_adhk_pdrb[df_adhk_pdrb['sector_code'] == 'PDRB']
    .groupby(['provinsi', 'year', 'quarter', 'period'])['value_billion_idr']
    .sum()
    .reset_index()
    .sort_values(['provinsi', 'year', 'quarter'])
)

pdrb_prov_yoy['pdrb_total_yoy'] = (
    pdrb_prov_yoy
    .groupby(['provinsi', 'quarter'])['value_billion_idr']
    .pct_change(fill_method=None) * 100
)

pdrb_sector_yoy = (
    df_adhk_pdrb[df_adhk_pdrb['sector_code'] != 'PDRB']
    .sort_values(['provinsi', 'sector_code', 'year', 'quarter'])
    .copy()
)

pdrb_sector_yoy['pdrb_sector_yoy'] = (
    pdrb_sector_yoy
    .groupby(['provinsi', 'sector_code', 'quarter'])['value_billion_idr']
    .pct_change(fill_method=None) * 100
)

pdrb_sector_yoy['sector_short'] = pdrb_sector_yoy['sector_name'].map(SECTOR_SHORT)

print('PDRB benchmarks ready.')
print('Latest periods:', sorted(pdrb_prov_yoy['period'].unique())[-4:])


PDRB benchmarks ready.
Latest periods: ['2025Q2', '2025Q3', '2025Q4', '2026Q1']


## 6. Plot functions

In [6]:
# ── Line chart: GVA YoY vs PDRB YoY (ADHK, quarterly) ───────────────────────
def plot_province_gva_yoy(province=province_list[0], top_n=8, all_n=17):
    """
    Panel 1: provincial total GVA YoY vs PDRB YoY (grouped bar, ADHK)
    Panel 2: top N volatile sectors — GVA YoY (solid) vs PDRB YoY (dashed)
    Panel 3: top all_n sectors (same but wider)
    X-axis: period (e.g. 2021Q1 ... 2026Q1), same-quarter YoY throughout.
    """
    prov_total = prov_adhk_qtr[
        (prov_adhk_qtr['provinsi'] == province) &
        (prov_adhk_qtr['gva_total_yoy'].notna())
    ].copy()

    sector_df = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['gva_yoy'].notna())
    ].copy()

    pdrb_bench = pdrb_prov_yoy[
        (pdrb_prov_yoy['provinsi'] == province) &
        (pdrb_prov_yoy['pdrb_total_yoy'].notna())
    ].copy()

    if prov_total.empty or sector_df.empty:
        print(f"No ADHK data for province '{province}'")
        return

    volatility_rank = (
        sector_df.groupby('sector_name')['gva_yoy']
        .mean().abs().sort_values(ascending=False).index.tolist()
    )

    def _subset(n):
        n = min(n, len(volatility_rank))
        sub = sector_df[sector_df['sector_name'].isin(volatility_rank[:n])].copy()
        pal = {SECTOR_SHORT[s]: SECTOR_COLOURS[SECTOR_SHORT[s]]
               for s in volatility_rank[:n] if SECTOR_SHORT.get(s) in SECTOR_COLOURS}
        pdrb_sub = pdrb_sector_yoy[
            (pdrb_sector_yoy['provinsi'] == province) &
            (pdrb_sector_yoy['sector_name'].isin(volatility_rank[:n])) &
            (pdrb_sector_yoy['pdrb_sector_yoy'].notna())
        ].copy()
        return sub, pal, pdrb_sub

    top_sectors, palette_top, pdrb_top = _subset(top_n)
    all_sectors, palette_all, pdrb_all = _subset(all_n)

    fig, axes = plt.subplots(3, 1, figsize=(16, 17), sharex=False)

    # Panel 1: grouped bar
    common_periods = sorted(set(prov_total['period'].values) & set(pdrb_bench['period'].values))
    gva_vals  = prov_total.set_index('period').loc[common_periods, 'gva_total_yoy'].values
    pdrb_vals = pdrb_bench.set_index('period').loc[common_periods, 'pdrb_total_yoy'].values
    x, w = np.arange(len(common_periods)), 0.35

    axes[0].bar(x - w/2, gva_vals,  w, color='#185FA5', label=f'{province} real GVA (ADHK)')
    axes[0].bar(x + w/2, pdrb_vals, w, color='#E63946', alpha=0.85,
                label=f'{province} real PDRB — BPS (ADHK)')
    for xi, v in zip(x - w/2, gva_vals):
        axes[0].text(xi, v + (0.05 if v >= 0 else -0.3), f'{v:.1f}%',
                     ha='center', va='bottom' if v >= 0 else 'top', fontsize=7, color='#185FA5')
    for xi, v in zip(x + w/2, pdrb_vals):
        axes[0].text(xi, v + (0.05 if v >= 0 else -0.3), f'{v:.1f}%',
                     ha='center', va='bottom' if v >= 0 else 'top', fontsize=7, color='#E63946')
    axes[0].axhline(0, color='grey', linestyle='--', linewidth=0.8)
    axes[0].set_title(f'{province} — real GVA vs PDRB YoY growth (ADHK, same-quarter)')
    axes[0].set_ylabel('YoY change (%)')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(common_periods, rotation=45, ha='right', fontsize=8)
    axes[0].legend(loc='best')

    # Panels 2 & 3
    for ax, sectors, palette, pdrb_sub, n_label in [
        (axes[1], top_sectors, palette_top, pdrb_top, top_n),
        (axes[2], all_sectors, palette_all, pdrb_all, all_n),
    ]:
        period_order_sub = sorted(sectors['period'].unique())
        for sector_s, group in pdrb_sub.groupby('sector_short'):
            colour = palette.get(sector_s, '#888780')
            group_sorted = group.sort_values('period')
            ax.plot(group_sorted['period'].values, group_sorted['pdrb_sector_yoy'].values,
                    color=colour, alpha=0.35, linewidth=1.2, linestyle='--', marker='s', markersize=4)
        for sector_s, group in sectors.groupby('sector_short'):
            colour = palette.get(sector_s, '#888780')
            group_sorted = group.sort_values('period')
            ax.plot(group_sorted['period'].values, group_sorted['gva_yoy'].values,
                    color=colour, linewidth=1.5, marker='o', markersize=4, label=sector_s)
        ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
        ax.set_title(f'{province} — top {n_label} sectors  (solid = GVA · dashed = PDRB, ADHK)')
        ax.set_ylabel('YoY (%)')
        ticks = period_order_sub[::2]
        ax.set_xticks(ticks)
        ax.set_xticklabels(ticks, rotation=45, ha='right', fontsize=8)
        ax.legend(title='Sector', bbox_to_anchor=(1.02, 1), loc='upper left',
                  fontsize=9 if n_label > 8 else 10)

    axes[2].set_xlabel('Period')
    plt.tight_layout()
    plt.show()


In [7]:
# ── Bar chart: sectoral GVA YoY by period (ADHK) ─────────────────────────────
def plot_province_gva_bar(province=province_list[0], periods=None):
    """
    Horizontal bar chart of sectoral GVA YoY growth (ADHK).
    Defaults to the 4 most recent periods.
    """
    if periods is None:
        periods = all_periods_qtr[-4:]

    sector_df = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['gva_yoy'].notna()) &
        (qtr_adhk['period'].isin(periods))
    ].copy()

    if sector_df.empty:
        print(f"No ADHK data for '{province}' in periods {periods}")
        return

    sector_order = (
        sector_df.groupby('sector_short')['gva_yoy']
        .mean().sort_values(ascending=True).index.tolist()
    )
    periods_sorted = sorted(periods, reverse=True)
    n_periods  = len(periods_sorted)
    n_sectors  = len(sector_order)
    bar_height = 0.8 / n_periods
    offsets    = [(i - (n_periods-1)/2) * bar_height for i in range(n_periods)]
    period_colours = {
        p: list(YEAR_COLOURS.values())[i % len(YEAR_COLOURS)]
        for i, p in enumerate(sorted(periods))
    }

    fig, ax = plt.subplots(figsize=(13, max(6, n_sectors * 0.55 + 1)))
    for i, (period, offset) in enumerate(zip(periods_sorted, offsets)):
        pr_df  = sector_df[sector_df['period'] == period].set_index('sector_short')
        colour = period_colours.get(period, '#888780')
        for j, sector in enumerate(sector_order):
            if sector not in pr_df.index: continue
            ax.barh(j + offset, pr_df.loc[sector, 'gva_yoy'],
                    height=bar_height * 0.9, color=colour,
                    edgecolor='white', linewidth=0.4,
                    label=period if j == 0 else '_nolegend_')

    ax.set_yticks(range(n_sectors))
    ax.set_yticklabels(sector_order, fontsize=10)
    ax.axvline(0, color='grey', linestyle='--', linewidth=1)
    ax.set_xlabel('YoY GVA growth (%, same quarter prior year)')
    ax.set_title(f'{province} — sectoral real GVA YoY growth (ADHK)')
    ax.legend(handles=[plt.Rectangle((0,0), 1, 1, fc=period_colours.get(p,'#888780'), label=p)
                        for p in periods_sorted],
              title='Period', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [8]:
# ── Bar chart: sectoral GVA as % of GDRP (ADHB) ──────────────────────────────
def plot_gva_pct_gdp(province=province_list[0]):
    """
    Sectoral GVA (basic prices, ADHB) as % of total provincial GDRP (market prices, ADHB).
    Correct basis for structural composition analysis.
    """
    gva_plot = ann_adhb_sorted[
        (ann_adhb_sorted['provinsi'] == province) &
        (ann_adhb_sorted['sector_code'] != 'PDRB')
    ][['sector_short', 'year', 'gva_pct_gdp']].copy()

    if gva_plot.empty:
        print(f"No ADHB data for '{province}'")
        return

    sector_order = (
        gva_plot.groupby('sector_short')['gva_pct_gdp']
        .mean().sort_values(ascending=True).index.tolist()
    )
    years_sorted = sorted(gva_plot['year'].unique(), reverse=True)
    n_years    = len(years_sorted)
    n_sectors  = len(sector_order)
    bar_height = 0.8 / n_years
    offsets    = [(i - (n_years-1)/2) * bar_height for i in range(n_years)]

    fig, ax = plt.subplots(figsize=(13, max(6, n_sectors * 0.55 + 1)))
    for i, (year, offset) in enumerate(zip(years_sorted, offsets)):
        yr_df  = gva_plot[gva_plot['year'] == year].set_index('sector_short')
        colour = YEAR_COLOURS.get(year, '#888780')
        for j, sector in enumerate(sector_order):
            if sector not in yr_df.index: continue
            ax.barh(j + offset, yr_df.loc[sector, 'gva_pct_gdp'],
                    height=bar_height * 0.9, color=colour,
                    edgecolor='white', linewidth=0.4,
                    label=str(year) if j == 0 else '_nolegend_')

    ax.set_yticks(range(n_sectors))
    ax.set_yticklabels(sector_order, fontsize=10)
    ax.set_xlabel('GVA as % of GDRP (current market prices, ADHB)')
    ax.set_title(f'{province} — sectoral GVA as % of GDRP  [ADHB structural view]')
    ax.legend(handles=[plt.Rectangle((0,0), 1, 1, fc=YEAR_COLOURS.get(y,'#888780'), label=str(y))
                        for y in years_sorted],
              title='Year', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [9]:
# ── Table: sectoral GVA YoY vs GDRP (ADHK, quarterly) ───────────────────────
def table_sector_vs_gdrp(province=province_list[0]):
    """
    Pivot: sectors × periods, GVA YoY (%, ADHK, same-quarter).
    GDRP reference row pinned at top.
    Green = above GDRP growth; red = below.
    """
    sect = (
        qtr_adhk[
            (qtr_adhk['provinsi'] == province) &
            (qtr_adhk['gva_yoy'].notna())
        ]
        [['sector_name', 'period', 'gva_yoy']].copy()
    )
    sect['sector_short'] = sect['sector_name'].map(SECTOR_SHORT)

    gdrp = (
        pdrb_prov_yoy[
            (pdrb_prov_yoy['provinsi'] == province) &
            pdrb_prov_yoy['pdrb_total_yoy'].notna()
        ]
        .set_index('period')['pdrb_total_yoy']
    )
    periods = sorted(gdrp.index)

    pivot = (
        sect.pivot(index='sector_short', columns='period', values='gva_yoy')
        .reindex(columns=periods)
    )
    pivot.columns.name = None
    pivot['Avg'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('Avg', ascending=False)

    gdrp_row = pd.DataFrame(
        [list(gdrp[periods]) + [gdrp[periods].mean()]],
        index=pd.Index(['GDRP (BPS, ADHK)'], name='sector_short'),
        columns=periods + ['Avg']
    )
    table = pd.concat([gdrp_row, pivot])

    gdrp_by_period = gdrp[periods].to_dict()
    gdrp_avg       = gdrp[periods].mean()

    def _highlight(row):
        if row.name == 'GDRP (BPS, ADHK)':
            return ['font-weight: bold; background-color: #e0e0e0'] * len(row)
        out = []
        for col in row.index:
            val = row[col]
            ref = gdrp_avg if col == 'Avg' else gdrp_by_period.get(col)
            if pd.isna(val) or ref is None:
                out.append(''); continue
            diff  = val - ref
            alpha = min(abs(diff) / 6, 0.65)
            colour = (f'rgba(45,106,79,{alpha:.2f})' if diff >= 0
                      else f'rgba(230,57,70,{alpha:.2f})')
            out.append(f'background-color: {colour}')
        return out

    return (
        table.style
        .apply(_highlight, axis=1)
        .format('{:.2f}%', na_rep='—')
        .set_caption(
            f'{province} — sectoral GVA YoY vs GDRP (ADHK, %)  ·  '
            f'green = above GDRP  ·  red = below')
    )


In [14]:
# ── Waterfall chart (ADHK) ──────────────────────────────────────────────
def plot_gva_waterfall(province=province_list[0], period=all_periods_qtr[-1]):
    """
    Horizontal waterfall — sectoral contribution in percentage points (YoY, ADHK).
    """
    current_q    = period[-2:]
    current_y    = int(period[:4])
    prior_period = f'{current_y - 1}{current_q}'

    curr = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['period'] == period)
    ].set_index('sector_name')

    prior = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['period'] == prior_period)
    ].set_index('sector_name')

    if curr.empty or prior.empty:
        print(f"Insufficient data for '{province}' — need both {period} and {prior_period}")
        return

    common = curr.index.intersection(prior.index)
    curr   = curr.loc[common]
    prior  = prior.loc[common]

    gva_total_prior = prior['gva_sector'].sum()

    contrib = ((curr['gva_sector'] - prior['gva_sector']) / gva_total_prior * 100)
    contrib = contrib.reset_index()
    contrib.columns = ['sector_name', 'contribution_pp']
    contrib['sector_short'] = contrib['sector_name'].map(SECTOR_SHORT)
    contrib = contrib.sort_values('contribution_pp', ascending=True).reset_index(drop=True)
    total_growth = contrib['contribution_pp'].sum()

    waterfall_colours = [
        SECTOR_COLOURS.get(s, '#888780') if v >= 0 else '#E63946'
        for s, v in zip(contrib['sector_short'], contrib['contribution_pp'])
    ]

    fig, ax = plt.subplots(figsize=(12, max(7, len(contrib) * 0.45 + 2)))

    bars = ax.barh(contrib['sector_short'], contrib['contribution_pp'],
                   color=waterfall_colours, edgecolor='white', linewidth=0.4)
    for bar, v in zip(bars, contrib['contribution_pp']):
        offset = 0.03 if v >= 0 else -0.03
        ax.text(v + offset, bar.get_y() + bar.get_height() / 2,
                f'{v:+.2f}pp', va='center',
                ha='left' if v >= 0 else 'right', fontsize=8)
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    ax.axvline(total_growth, color='#185FA5', linestyle='-', linewidth=1.5,
               label=f'Total GVA YoY: {total_growth:+.2f}pp')
    ax.set_xlabel('Contribution to GVA growth (percentage points)')
    ax.set_title(
        f'{province} — sectoral contribution to GVA growth\n'
        f'{prior_period} → {period}  (ADHK, same-quarter YoY)',
        fontsize=12, fontweight='bold'
    )
    ax.legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Contribution pie chart (ADHK) ───────────────────────────────────────
def plot_gva_pie(province=province_list[0], period=all_periods_qtr[-1],
                 bundle_threshold=0.5):
    """
    Pie — share of total absolute growth each sector contributed (YoY, ADHK).
    Sectors with |contribution| < bundle_threshold pp are bundled into 'Other'.
    """
    current_q    = period[-2:]
    current_y    = int(period[:4])
    prior_period = f'{current_y - 1}{current_q}'

    curr = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['period'] == period)
    ].set_index('sector_name')

    prior = qtr_adhk[
        (qtr_adhk['provinsi'] == province) &
        (qtr_adhk['period'] == prior_period)
    ].set_index('sector_name')

    if curr.empty or prior.empty:
        print(f"Insufficient data for '{province}' — need both {period} and {prior_period}")
        return

    common = curr.index.intersection(prior.index)
    curr   = curr.loc[common]
    prior  = prior.loc[common]

    gva_total_prior = prior['gva_sector'].sum()

    contrib = ((curr['gva_sector'] - prior['gva_sector']) / gva_total_prior * 100)
    contrib = contrib.reset_index()
    contrib.columns = ['sector_name', 'contribution_pp']
    contrib['sector_short'] = contrib['sector_name'].map(SECTOR_SHORT)
    contrib['abs_contrib']  = contrib['contribution_pp'].abs()

    main    = contrib[contrib['abs_contrib'] >= bundle_threshold].copy()
    bundled = contrib[contrib['abs_contrib'] <  bundle_threshold].copy()

    pie_data = main[['sector_short', 'contribution_pp', 'abs_contrib']].copy()
    if not bundled.empty:
        pie_data = pd.concat([pie_data, pd.DataFrame([{
            'sector_short'   : 'Other',
            'contribution_pp': bundled['contribution_pp'].sum(),
            'abs_contrib'    : bundled['abs_contrib'].sum(),
        }])], ignore_index=True)
    pie_data = pie_data.sort_values('abs_contrib', ascending=False).reset_index(drop=True)

    def _colour(row):
        if row['sector_short'] == 'Other': return '#CCCCCC'
        if row['contribution_pp'] >= 0: return SECTOR_COLOURS.get(row['sector_short'], '#888780')
        return '#E63946'

    pie_colours = [_colour(r) for _, r in pie_data.iterrows()]

    fig, ax = plt.subplots(figsize=(9, 7))

    explode = [0.05 if v < 0 else 0 for v in pie_data['contribution_pp']]
    wedges, texts, autotexts = ax.pie(
        pie_data['abs_contrib'],
        labels=None, colors=pie_colours, explode=explode,
        autopct=lambda p: f'{p:.1f}%' if p >= 3 else '',
        pctdistance=0.75, startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 0.8}
    )
    for at in autotexts: at.set_fontsize(8)

    legend_labels = [f"{r['sector_short']}  ({r['contribution_pp']:+.2f}pp)"
                     for _, r in pie_data.iterrows()]
    ax.legend(wedges, legend_labels,
              title='Sector  (negative = dragged growth)',
              loc='upper left', bbox_to_anchor=(-0.35, -0.05),
              fontsize=8, title_fontsize=8, frameon=True)
    ax.set_title(
        f'{province} — share of absolute GVA growth movement\n'
        f'{prior_period} → {period}  (ADHK, same-quarter YoY)',
        fontsize=12, fontweight='bold'
    )

    plt.tight_layout()
    plt.show()

## 7. Launch

In [15]:
DEMO_PROVINCE = province_list[0]

if widget_support:
    print('── Line chart (ADHK) ──')
    interact(plot_province_gva_yoy,
             province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'),
             top_n=widgets.IntSlider(min=2, max=17, step=1, value=8,  description='Panel 2 N:'),
             all_n=widgets.IntSlider(min=2, max=17, step=1, value=17, description='Panel 3 N:'))

    print('── Bar chart: GVA YoY (ADHK) ──')
    interact(plot_province_gva_bar,
             province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'),
             periods=widgets.SelectMultiple(options=all_periods_qtr, value=tuple(all_periods_qtr[-4:]),
                                            description='Periods:', rows=6))

    print('── Bar chart: GVA % of GDRP (ADHB structural view) ──')
    interact(plot_gva_pct_gdp,
             province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'))

    print('── Table: sector vs GDRP (ADHK) ──')
    interact(table_sector_vs_gdrp,
             province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'))

    print('── Waterfall: sectoral growth contributions (ADHK) ──')
    interact(plot_gva_waterfall,
            province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'),
            period=widgets.Dropdown(options=all_periods_qtr[1:], value=all_periods_qtr[-1], description='Period:'))

    print('── Pie: share of absolute GVA growth movement (ADHK) ──')
    interact(plot_gva_pie,
            province=widgets.Dropdown(options=province_list, value=DEMO_PROVINCE, description='Province:'),
            period=widgets.Dropdown(options=all_periods_qtr[1:], value=all_periods_qtr[-1], description='Period:'),
            bundle_threshold=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5,
                                    description='Bundle <', readout_format='.1f'))


── Line chart (ADHK) ──


interactive(children=(Dropdown(description='Province:', options=('Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yog…

── Bar chart: GVA YoY (ADHK) ──


interactive(children=(Dropdown(description='Province:', options=('Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yog…

── Bar chart: GVA % of GDRP (ADHB structural view) ──


...

interactive(children=(Dropdown(description='Province:', options=('Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yog…

── Waterfall: sectoral growth contributions (ADHK) ──


interactive(children=(Dropdown(description='Province:', options=('Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yog…

── Pie: share of absolute GVA growth movement (ADHK) ──


interactive(children=(Dropdown(description='Province:', options=('Aceh', 'Bali', 'Banten', 'Bengkulu', 'DI Yog…